---

«Ya sabemos cómo utilizar el método `generate`. Sin embargo, los modelos que descargamos de Hugging Face cuentan con ajustes y parámetros específicos que pueden arrojar errores ante entradas faltantes o situaciones particulares. Por lo tanto, es fundamental recordar que la sintaxis para interactuar con ellos varía en función de la arquitectura de cada modelo».

---

El token de fin de secuencia, conocido como **EOS (End-of-Sentence)**, juega un papel fundamental en los modelos de lenguaje basados en transformadores (como GPT-2 en Hugging Face).

### ¿Qué es y cómo funciona el token EOS?

El token EOS (con el ID `50256` en el caso de GPT-2) es un marcador especial que el modelo aprende a reconocer durante su preentrenamiento para indicar que una oración o un fragmento de texto ha llegado a su conclusión natural.

### ¿Cómo detiene la generación y cómo lo hace internamente?

1. **Detección iterativa:** Durante la generación autorregresiva, el modelo predice los tokens uno por uno de forma secuencial. En cada paso, evalúa la probabilidad de todos los tokens del vocabulario, incluyendo el token EOS.

2. **Condición de parada:** El método `generate()` de Hugging Face vigila activamente la salida. Si el modelo predice o genera el token EOS, el bucle de generación interno interpreta que la secuencia ha terminado.

3. **Terminación de la secuencia:** En ese momento, el proceso de muestreo se detiene para esa secuencia específica, incluso si no se ha alcanzado el límite máximo de tokens (`max_length`), evitando que el modelo comience a generar texto basura, repeticiones infinitas o divagaciones sin sentido.


In [1]:
#Esto nos sirve para generar multiples sentencias con diferentes longitudes de secuencia

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer  #
import torch  #

# Cargar el tokenizador y el modelo preentrenados de GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")  # podemos colocar diferentes modelos gpt2-small etc
model = GPT2LMHeadModel.from_pretrained("gpt2")  #

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

---

«Configurar el token de relleno (*pad token*) como el token EOS (*end-of-sentence*) puede generar cierta confusión en el modelo al introducir múltiples delimitadores de fin de secuencia intercalados en el texto. Sin embargo, durante el *fine-tuning*, los textos largos suelen concatenarse o truncarse en fragmentos de longitud fija, por lo que este solapamiento no representa un problema crítico para el entrenamiento. Del mismo modo, en la fase de evaluación funciona adecuadamente, ya que por lo general los marcadores EOS se ubican de forma natural al final de las secuencias».

---

In [3]:
# GPT-2 no tiene un token de relleno (pad token) predeterminado[cite: 12]
print("Current pad token: ", tokenizer.pad_token)  #[cite: 12]
print("Current eos token: ", tokenizer.eos_token)  #[cite: 12]

# Es común (aunque no necesariamente ideal) configurar el token de relleno como el token EOS[cite: 12]
tokenizer.pad_token = tokenizer.eos_token  #[cite: 12]
model.config.pad_token_id = tokenizer.eos_token_id  #[cite: 12] se puede cambiar por un espacio o /t etc

print("Current pad token: ", tokenizer.pad_token)  #[cite: 12] estamos configurandolo para que no sea none

Current pad token:  None
Current eos token:  <|endoftext|>
Current pad token:  <|endoftext|>


Encodear

In [4]:
sentences = [
    "Every happy family is the same",
    "every unhappy family is unhappy it its own way",
    "Why did the chicken cross the road?",
]  #[cite: 13]

# tokenizar con relleno (padding) y máscara de atención
toks = tokenizer(
    sentences, return_tensors="pt", padding=True
)  #[cite: 13]

toks  #[cite: 13]

{'input_ids': tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835],
        [ 5195,   750,   262,  9015,  3272,   262,  2975,    30, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])}

---

«Este bloque nos muestra las secuencias de tokens para las diferentes oraciones, junto con los tokens de relleno (*padding tokens*) y la máscara de atención (*attention mask*). Las posiciones que reciben un valor de cero en la máscara indican que Hugging Face está configurado para omitir y no procesar dichos tokens durante los cálculos del modelo».

---

In [5]:
for i in range(len(sentences)):  #[cite: 14]

  inputs = toks["input_ids"][i]  #[cite: 14]
  mask = toks["attention_mask"][i]  #[cite: 14]

  print(f'"{sentences[i]}":')  #[cite: 14]
  print(inputs, "\n", mask, "\n")  #[cite: 14]

"Every happy family is the same":
tensor([ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256]) 
 tensor([1, 1, 1, 1, 1, 1, 0, 0, 0]) 

"every unhappy family is unhappy it its own way":
tensor([16833, 19283,  1641,   318, 19283,   340,   663,   898,   835]) 
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]) 

"Why did the chicken cross the road?":
tensor([ 5195,   750,   262,  9015,  3272,   262,  2975,    30, 50256]) 
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 0]) 



Generar algo de texto

no es necesario usar todas estas opciones por lo general se utilizan top_k y top_p

In [8]:
outputs = model.generate( # anteriormetne lo generamos solo con los (tokens) pero el metodo generate tiene varias opciones
    input_ids=toks["input_ids"], # inputs ids
    attention_mask=toks["attention_mask"], # la mascara necesaria si son diferentes longitudes
    pad_token_id=tokenizer.pad_token_id,
    max_length=66, # la maxima longitud que incluye la max longitud de la entrada, en la salida el modelo puede detenerse antes no necesariamente serian 66 por ejemplo
    num_return_sequences=1, # solo queremos una secuencia de vuelta
    do_sample=True, # el modelo seleccioanra probabilisticamente los tokens con mas estocasticidad
    top_k=50, #top de los 50 mejores logits
    top_p=0.95, # que solo den una probabilidad del 95% seran seleccionadas
)  #[cite: 15]

outputs  #[cite: 15]

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256,    11,
           772,   257,  1641,   286,  1440,   743,  1064,  2405,   287,   257,
          5802,  3074,    13,  1881,  1545,   373,  6686,   287,   257,  1097,
          5778,    13,  1119,  1816,   284,   257,  4436,   290,   423,  1201,
          4504,   284,   511,  1641,    11,   475,   407,   257,  2060,   530,
           468,   587,  5938,   287,   511,  3160,   526,   198,   198,    47,
           945,   684,   531,   262,  2055,   318],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835,    13,
           317,  1256,   286,  4172,  1064,  2405,   287,  8098,   290,  1231,
           597,  1037,    11,   340,   338,  1327,   329,   606,   284,  1414,
           329,  1997,    13,   843,   788,   340,  1838,   477,   777,   661,
          1254,   588,   484,   821,  2263,   510,   511,   898,  3160,    11,
           290,   523,   484,  1254,   588,  3863,   617,   584,  1048,   481,


Decodificar

In [10]:
outputs.shape

torch.Size([3, 66])

In [9]:
# Decodificar las salidas generadas
decoded_texts = tokenizer.batch_decode( # cuando nos encontramos con esta situacion torch.Size([3, 66]) utilizamos batch_decode en vez de decode
    outputs, skip_special_tokens=True # omitir los tokens especiales
)  #[cite: 16]

for i, text in enumerate(decoded_texts):  #[cite: 16]
  print(f"*** Text {i+1}:\n{text}\n")  #[cite: 16]

*** Text 1:
Every happy family is the same, even a family of four may find themselves in a tough situation. One friend was injured in a car accident. They went to a hospital and have since returned to their family, but not a single one has been hurt in their lives."

Parsons said the community is

*** Text 2:
every unhappy family is unhappy it its own way. A lot of families find themselves in poverty and without any help, it's hard for them to pay for anything. And then it makes all these people feel like they're taking up their own lives, and so they feel like maybe some other person will step in and help them out

*** Text 3:
Why did the chicken cross the road? the lady in the suit looked at me skeptically, then said 'OK, you're going to have to go back,' and there was nothing to do but sit on the porch and stare. And then she picked me up and gave me a warm smile. The lady was completely stunned



In [11]:
# O de forma más simple:
model.generate(toks["input_ids"])

[transformers] The attention mask and the pad token id were not set, with a batched input. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=29) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256,   464,
           471,    13,    50,    13,  2732,   286,  4796,   468,  5717,   257,
          8087,  1028,   262,   471,    13,    50,    13,  2732,   286],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835,    13,
           198,   198,     1,    40,   892,   340,   338,   257,   845,  1593,
          2071,   329,   262,  1499,    13,   632,   338,   257,   845],
        [ 5195,   750,   262,  9015,  3272,   262,  2975,    30, 50256,   464,
           717,   640,   314,  2497,   262,   649,   366,   464, 21276,  5542,
             1, 12268,    11,   314,   373,   523,  6568,    13,   314]])

### ¿Qué ocurre en este punto?

* **Advertencias automáticas:** Al llamar al método `model.generate()` pasando únicamente `toks['input_ids']` sin la máscara de atención ni el identificador de relleno explícitos, Hugging Face detecta la omisión y emite advertencias (*warnings*).


* **Configuración por defecto:** El framework asigna automáticamente el token EOS (`50256`) como el `pad_token_id` para la generación de texto abierto, advirtiendo sobre posibles comportamientos inesperados debido al uso de relleno a la derecha (*right-padding*) en una arquitectura exclusiva de decodificador (*decoder-only*).

###  Conclusión Clave: Más allá de la sintaxis

La experiencia con Hugging Face nos demuestra que **la sintaxis no es universal**; cada arquitectura de modelo maneja particularidades distintas en cuanto a parámetros, gestión de relleno (*padding*) y máscaras de atención.

Por esta razón, intentar memorizar líneas de código específicas suele ser ineficiente. El verdadero criterio técnico se desarrolla al **explorar y comprender la estructura de las entradas y salidas** (`input_ids`, `attention_mask`, tensores generados). Al analizar cómo viajan los datos y qué espera el modelo en cada paso.